### Implementar simulação de predição por sessão (loop unitário com dados recentes).

#### Imports, parâmetros e MLflow

In [1]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import mlflow
import mlflow.sklearn
import mlflow.tensorflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

# Caminhos / artefatos
DATA_PATH = "dados/df_holdout.csv"  # contém Revenue (target)
RUN_MLP_UNDER = "5c59bad245764d43b54d3bd66249670b"  # mlp__under
RUN_RF_UNDER  = "7bd415c0b41141fd9cf0fbe881c2493b"  # under__rf
RUN_XGB_UNDER = "a15d0102fc1042b3857ede82117fec81"  # under__xgb

# MLflow (aponta para sua pasta local mlruns)
MLRUNS_DIR = Path("mlruns").resolve()
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
mlflow.set_experiment("production")
client = MlflowClient()
print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI: file:///D:/Drive/Academico/DataScience_XP/ProjetoAplicado/projetoAplicadoDSXP/mlruns


#### Função de codificação (a sua, 1:1)

In [2]:
# =============================
# 0) Função de codificação híbrida
# =============================
def encode_features(df_in):
    df = df_in.copy()

    # (a) Month -> ordinal (ordem temporal)
    meses_ordem = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_map = {m:i+1 for i,m in enumerate(meses_ordem)}  # Feb=1, Mar=2, ... Dec=10
    df['Month'] = df['Month'].map(month_map).astype('int64')

    # (b) VisitorType -> One-Hot (base = New)
    df['VisitorType'] = df['VisitorType'].replace({
        'Returning_Visitor': 'Returning',
        'New_Visitor': 'New',
        'Other': 'Other'
    })
    df = pd.get_dummies(df, columns=['VisitorType'], drop_first=True)
    # Cria colunas: VisitorType_Other, VisitorType_Returning (base implícita = New)

    # (c) Weekend -> garantir inteiro
    if df['Weekend'].dtype != 'int64' and df['Weekend'].dtype != 'int32':
        df['Weekend'] = df['Weekend'].astype(int)

    # Todas as demais já são numéricas
    return df


#### Carregar holdout, aplicar encoding e separar X / y

In [3]:
dataset_original = pd.read_csv(DATA_PATH, encoding="utf-8")
df_enc = encode_features(dataset_original)

has_target = 'Revenue' in df_enc.columns
if has_target:
    y_true_all = df_enc['Revenue'].astype(int).values
    X_df = df_enc.drop(columns=['Revenue']).copy()
else:
    y_true_all = None
    X_df = df_enc.copy()

print("Shape X_df:", X_df.shape, "| target presente?", has_target)
print("Colunas:", list(X_df.columns))


Shape X_df: (345, 23) | target presente? True
Colunas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'PageValues_Outlier', 'ProductRelated_Duration_Outlier', 'ProductPageViewRate', 'ExitBounceRatio', 'ViewedProduct', 'VisitorType_Other', 'VisitorType_Returning']


#### Utils (alinhamento de colunas, scaler e simulador)

In [4]:
def align_features_for_model(X_df: pd.DataFrame, model) -> pd.DataFrame:
    """
    Reordena/seleciona as colunas para bater com o esquema do modelo (se disponível).
    Cria colunas faltantes com 0 e remove extras.
    """
    cols_expected = getattr(model, "feature_names_in_", None)
    if cols_expected is None:
        return X_df
    X_aligned = X_df.copy()
    for c in cols_expected:
        if c not in X_aligned.columns:
            X_aligned[c] = 0
    return X_aligned[list(cols_expected)]

def is_keras_model(model) -> bool:
    try:
        import inspect
        return 'verbose' in dict(inspect.signature(model.predict).parameters)
    except Exception:
        return False

def standardize_with_saved_scaler(X_np: np.ndarray, mean: np.ndarray, scale: np.ndarray) -> np.ndarray:
    safe_scale = np.where(scale == 0, 1.0, scale)
    return (X_np - mean) / safe_scale

def simulate_stream(model, model_name: str, X_df: pd.DataFrame, y_true=None,
                    scaler_mean=None, scaler_scale=None, batch_print: int = 200):
    preds, probs, truths, idxs = [], [], [], []
    n = X_df.shape[0]

    X_model = align_features_for_model(X_df, model)
    keras_flag = is_keras_model(model)

    for i in range(n):
        if keras_flag:
            x_np = X_model.iloc[i:i+1].to_numpy(dtype=float)
            if scaler_mean is not None and scaler_scale is not None:
                if x_np.shape[1] == scaler_mean.shape[0] == scaler_scale.shape[0]:
                    x_np = standardize_with_saved_scaler(x_np, scaler_mean, scaler_scale)
                else:
                    raise ValueError(f"MLP ignorada: X={x_np.shape[1]} vs scaler={scaler_mean.shape[0]}")
            p = float(model.predict(x_np, verbose=0).ravel()[0])
        else:
            p = float(model.predict_proba(X_model.iloc[i:i+1])[0, 1])

        yhat = int(p >= 0.5)
        preds.append(yhat); probs.append(p); idxs.append(i)
        if y_true is not None:
            truths.append(int(y_true[i]))

        if y_true is not None and (i+1) % batch_print == 0:
            acc = accuracy_score(truths, preds)
            prec = precision_score(truths, preds, zero_division=0)
            rec  = recall_score(truths, preds, zero_division=0)
            f1   = f1_score(truths, preds, zero_division=0)
            try:
                auc = roc_auc_score(truths, probs)
            except Exception:
                auc = np.nan
            print(f"[{model_name}] {i+1}/{n} -> acc={acc:.3f} | prec={prec:.3f} | rec={rec:.3f} | f1={f1:.3f} | auc={auc:.3f}")

    log_df = pd.DataFrame({"idx": idxs, "y_pred": preds, "y_prob": probs})
    metrics = {}
    if y_true is not None and len(truths) > 0:
        metrics["acc"]  = accuracy_score(truths, preds)
        metrics["prec"] = precision_score(truths, preds, zero_division=0)
        metrics["rec"]  = recall_score(truths, preds, zero_division=0)
        metrics["f1"]   = f1_score(truths, preds, zero_division=0)
        try:
            metrics["auc"] = roc_auc_score(truths, probs)
        except Exception:
            metrics["auc"] = np.nan
    return log_df, metrics


#### Carregar modelos e scaler (MLflow)

In [5]:
# MLP (Keras): 'keras_model'
model_mlp = mlflow.tensorflow.load_model(f"runs:/{RUN_MLP_UNDER}/keras_model")

# RF (sklearn): 'model'
model_rf  = mlflow.sklearn.load_model(f"runs:/{RUN_RF_UNDER}/model")

# XGB: 'model' (se falhar, pyfunc)
try:
    model_xgb = mlflow.sklearn.load_model(f"runs:/{RUN_XGB_UNDER}/model")
except Exception:
    model_xgb = mlflow.pyfunc.load_model(f"runs:/{RUN_XGB_UNDER}/model")

# Scaler salvo do MLP (se existir)
def try_load_mlp_scaler(run_id: str):
    try:
        m = np.load(client.download_artifacts(run_id, "scaler_mean.npy"))
        s = np.load(client.download_artifacts(run_id, "scaler_scale.npy"))
        return m, s
    except Exception:
        return None, None

scaler_mean, scaler_scale = try_load_mlp_scaler(RUN_MLP_UNDER)
print("Scaler MLP:", None if scaler_mean is None else scaler_mean.shape)


Scaler MLP: (23,)


#### Rodar simulação dos 3 modelos (usando encode_features)

In [7]:
results = {}

# MLP (só roda se nº de colunas bater com scaler)
try:
    if scaler_mean is not None and scaler_scale is not None:
        if X_df.shape[1] == scaler_mean.shape[0] == scaler_scale.shape[0]:
            log_mlp, met_mlp = simulate_stream(
                model_mlp, "mlp__under", X_df, y_true_all,
                scaler_mean=scaler_mean, scaler_scale=scaler_scale
            )
            results["mlp__under"] = (log_mlp, met_mlp)
        else:
            print(f"⚠️ MLP ignorada: X_df tem {X_df.shape[1]} colunas e scaler tem {scaler_mean.shape[0]}.")
    else:
        print("⚠️ MLP ignorada: scaler não encontrado nos artefatos.")
except Exception as e:
    print("⚠️ MLP ignorada por erro:", e)

# RF e XGB
log_rf,  met_rf  = simulate_stream(model_rf,  "under__rf",  X_df, y_true_all)
log_xgb, met_xgb = simulate_stream(model_xgb, "under__xgb", X_df, y_true_all)

results["under__rf"]  = (log_rf,  met_rf)
results["under__xgb"] = (log_xgb, met_xgb)

print("\n=== Métricas finais (holdout) ===")
for name, (_, m) in results.items():
    print(f"{name}:", m)

# salvar predições
Path("dados").mkdir(parents=True, exist_ok=True)
if "mlp__under" in results:
    results["mlp__under"][0].to_csv("dados/predicoes_mlp__under.csv", index=False)
results["under__rf"][0].to_csv("dados/predicoes_under__rf.csv", index=False)
results["under__xgb"][0].to_csv("dados/predicoes_under__xgb.csv", index=False)
print("Arquivos salvos em dados/*.csv")

[mlp__under] 200/345 -> acc=0.800 | prec=0.426 | rec=0.839 | f1=0.565 | auc=0.867
[under__rf] 200/345 -> acc=0.785 | prec=0.406 | rec=0.839 | f1=0.547 | auc=0.871
[under__xgb] 200/345 -> acc=0.795 | prec=0.419 | rec=0.839 | f1=0.559 | auc=0.888

=== Métricas finais (holdout) ===
mlp__under: {'acc': 0.8057971014492754, 'prec': 0.40594059405940597, 'rec': 0.8541666666666666, 'f1': 0.5503355704697986, 'auc': 0.891905162738496}
under__rf: {'acc': 0.7971014492753623, 'prec': 0.39215686274509803, 'rec': 0.8333333333333334, 'f1': 0.5333333333333333, 'auc': 0.8766133557800225}
under__xgb: {'acc': 0.7913043478260869, 'prec': 0.38461538461538464, 'rec': 0.8333333333333334, 'f1': 0.5263157894736842, 'auc': 0.8848204264870932}
Arquivos salvos em dados/*.csv


### Consolidar os logs e versões no MLflow.

#### Logar simulação no MLflow (production)

In [8]:
# ===== Bloco 6C: Logar simulação no MLflow =====
from pathlib import Path

LOG_TO_MLFLOW = True  # coloque False se não quiser logar

# Mapeia o nome do modelo da simulação para o run_id original (o que você me passou)
SOURCE_RUNS = {
    "mlp__under": RUN_MLP_UNDER,
    "under__rf":  RUN_RF_UNDER,
    "under__xgb": RUN_XGB_UNDER,
}

# (opcional) caminhos dos CSVs salvos no Bloco 5
artifact_paths = {
    "mlp__under": "dados/predicoes_mlp__under.csv",
    "under__rf":  "dados/predicoes_under__rf.csv",
    "under__xgb": "dados/predicoes_under__xgb.csv",
}

if LOG_TO_MLFLOW:
    for name, (log_df, metrics) in results.items():
        run_name = f"prod_sim__{name}"
        src_run  = SOURCE_RUNS.get(name, "unknown")
        csv_path = artifact_paths.get(name)

        with mlflow.start_run(run_name=run_name, description="Simulação holdout/produção") as run:
            # tags úteis para rastreabilidade
            mlflow.set_tags({
                "phase": "production_sim",
                "source_run_id": src_run,
                "model_name": name,
                "dataset": "holdout_dezembro",
                "encoding_fn": "encode_features_v1",   # sua função
            })

            # parâmetros informativos
            mlflow.log_params({
                "holdout_rows": len(X_df),
                "target_present": bool(y_true_all is not None),
                "tracking_uri": mlflow.get_tracking_uri(),
            })

            # métricas finais desta simulação
            for k, v in metrics.items():
                if v is not None:
                    mlflow.log_metric(k, float(v))

            # artefatos: predicoes como CSV (se existir)
            if csv_path and Path(csv_path).exists():
                mlflow.log_artifact(csv_path, artifact_path="predicoes")

        print(f"✔ Logado no MLflow: {run_name}  (source_run_id={src_run})")
else:
    print("LOG_TO_MLFLOW=False -> não foi feito log no MLflow.")
#$ mlflow ui --backend-store-uri file:///C:/SEU/CAMINHO/para/mlruns

✔ Logado no MLflow: prod_sim__mlp__under  (source_run_id=5c59bad245764d43b54d3bd66249670b)
✔ Logado no MLflow: prod_sim__under__rf  (source_run_id=7bd415c0b41141fd9cf0fbe881c2493b)
✔ Logado no MLflow: prod_sim__under__xgb  (source_run_id=a15d0102fc1042b3857ede82117fec81)
